# 02 — EV Projection Fork (MANDATORY)

Run the datos.gob.es Exercise 3 EV growth model. Extract the 2027 projected EV fleet number for Spain.

**This is MANDATORY** — the output feeds directly into File 1 and the demand model.

## Data Inputs
- `data/raw/datos_gob_ev_forecast/` — parquet files from datos.gob.es GitHub fork
- DGT vehicle registration parquet files (2015-2023)

## Data Outputs
- `data/processed/ev_projection_2027.csv` — EV fleet projection by year
- `total_ev_projected_2027` scalar value (used in File_1.csv)

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), ''))
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

PROCESSED = Path('data/processed')

# Load the cleaned monthly EV registration time series from Notebook 01
ev = pd.read_csv(PROCESSED / 'ev_monthly_registrations.csv', parse_dates=['date'])
ev = ev.set_index('date')['ev_registrations']
print(f"Monthly BEV+PHEV registrations: {len(ev)} months ({ev.index.min()} – {ev.index.max()})")
print(f"Total registrations: {ev.sum():,}")
print(f"\nLast 6 months:")
print(ev.tail(6))

plt.figure(figsize=(12, 4))
plt.plot(ev.index, ev.values)
plt.title('Monthly BEV+PHEV New Registrations in Spain (2015–2023)')
plt.ylabel('Registrations')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## SARIMA Model Fitting

We use `pmdarima.auto_arima` on the log-transformed series to capture the exponential growth trend and seasonal patterns (12-month cycle). This follows the methodology from the datos.gob.es Exercise 3 fork.

In [ ]:
import pmdarima as pm
from statsmodels.tsa.stattools import adfuller

# Log-transform (handles exponential growth better)
log_ev = np.log(ev.values)

# ADF test on log series
adf_result = adfuller(log_ev)
print(f"ADF Statistic: {adf_result[0]:.4f}")
print(f"p-value: {adf_result[1]:.4f}")
print(f"Series is {'stationary' if adf_result[1] < 0.05 else 'non-stationary (needs differencing)'}")

# Fit SARIMA with auto_arima
print("\nFitting SARIMA model (this may take a minute)...")
model = pm.auto_arima(
    log_ev,
    seasonal=True,
    m=12,
    test='adf',
    stepwise=True,
    trace=True,
    error_action='ignore',
    suppress_warnings=True,
)

print(f"\nBest model: {model.order} x {model.seasonal_order}")
print(model.summary())

## Forecast 2024–2027

Forecast 48 months forward (Jan 2024 – Dec 2027). Convert back from log scale. Compute cumulative EV fleet to get `total_ev_projected_2027`.

In [ ]:
# Forecast 48 months ahead (2024-01 to 2027-12)
n_forecast = 48
log_forecast, conf_int = model.predict(n_periods=n_forecast, return_conf_int=True)

# Convert from log scale
forecast = np.exp(log_forecast)
ci_lower = np.exp(conf_int[:, 0])
ci_upper = np.exp(conf_int[:, 1])

# Create forecast dates
forecast_dates = pd.date_range(start='2024-01-01', periods=n_forecast, freq='MS')

# Combine actual + forecast
full_series = pd.DataFrame({
    'date': list(ev.index) + list(forecast_dates),
    'ev_registrations': list(ev.values) + list(forecast.astype(int)),
    'type': ['actual'] * len(ev) + ['forecast'] * n_forecast,
})
full_series['cumulative_ev_fleet'] = full_series['ev_registrations'].cumsum()

# Extract the key number
total_ev_projected_2027 = int(full_series['cumulative_ev_fleet'].iloc[-1])
print(f"total_ev_projected_2027 = {total_ev_projected_2027:,}")
print(f"\nForecast summary (annual new registrations):")
for year in [2024, 2025, 2026, 2027]:
    mask = full_series['date'].dt.year == year
    print(f"  {year}: {full_series.loc[mask, 'ev_registrations'].sum():,}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
actual = full_series[full_series['type'] == 'actual']
fcast = full_series[full_series['type'] == 'forecast']
ax.plot(actual['date'], actual['ev_registrations'], label='Actual', color='blue')
ax.plot(fcast['date'], fcast['ev_registrations'], label='Forecast', color='red', linestyle='--')
ax.fill_between(forecast_dates, ci_lower, ci_upper, alpha=0.2, color='red')
ax.set_title('Monthly BEV+PHEV Registrations — Actual vs Forecast')
ax.set_ylabel('Registrations')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(full_series['date'], full_series['cumulative_ev_fleet'])
ax.axvline(x=pd.Timestamp('2027-12-31'), color='red', linestyle=':', label='2027 target')
ax.set_title(f'Cumulative EV Fleet → {total_ev_projected_2027:,} by 2027')
ax.set_ylabel('Cumulative Fleet')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Save Projection Output

In [ ]:
# Save the full projection
full_series.to_csv(PROCESSED / 'ev_projection_2027.csv', index=False)
print(f"Saved: ev_projection_2027.csv ({full_series.shape[0]} rows)")
print(f"\ntotal_ev_projected_2027 = {total_ev_projected_2027:,}")
print("This value goes into File 1 (Global Network KPIs)")

# Verify the output structure
print(f"\nOutput structure:")
print(full_series.info())
print(f"\nLast 12 months of forecast:")
full_series.tail(12)